In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import subprocess
from numba import jit

In [ ]:
dt=0.01
the0=[1.,-1.]
t_episode=20

In [ ]:
def RHS(x,ome=1,mu=1):
    f1=x[1]*ome
    f2=-x[0]*ome-ome*mu*x[1]*(x[1]**2-4*np.pi**2)
    if x[1]>2*np.pi: f2=-0.1
    if x[1]<-2*np.pi: f2=0.1
    return np.array([f1,f2])

def orbit(ome,mu,ns,dt,x0=[0,2*np.pi]):
    x=np.zeros((ns+1,2),dtype=np.float64)
    x[0]=x0
    for n in range(ns):
        f0=RHS(x[n],ome=ome[n],mu=mu[n])
        tmp=x[n]+f0*dt
        x[n+1]=x[n]+(f0+RHS(tmp,ome=ome[n],mu=mu[n]))*dt/2
    return x

def period(x,dt):
    vmax=2.
    imax=0
    imaxs=[]
    for i in range(1,len(x)):
        if x[-i-1,1]>x[-i,1] and x[-i,1]>2:
            vmax=x[-i-1,1]
            imax=-i-1
        if imax<0 and x[-i,1]<0.:
            imaxs.append(imax)
            imax=0
        if len(imaxs)>=2: break
    return (imaxs[0]-imaxs[1])*dt

ts=np.arange(0.,100+dt,dt,dtype=np.float64)
with open("omega0.txt", "w") as f:
    omega=np.zeros(len(ts),dtype=np.float64)
    for i in range(len(ts)):
        omega[i]=np.pi*2*np.sin(np.pi*ts[i])
        f.write(f'{ts[i]} {omega[i]} \n')
omega

result = subprocess.run(['cp','input0.nml','input.nml'])
result = subprocess.run(['../../src/a.out'], capture_output=True, text=True)
print(result.stdout)
df = pd.read_csv('./axke.dat', sep="\s+",header=None, skiprows=1)
plt.plot(df.iloc[:,1].values)
plt.show()

dt_limit=0.0001
nt_limit=round(t_episode/dt_limit)+1
ome0,mu0,x0=the0[0],the0[1],[0,2*np.pi]
result = subprocess.run(['cp','input1.nml','input.nml'])
for n in range(epoch):
    with open("input.nml", "r") as f:
        content = f.read()
    new_content = content.replace(f"tend = {10.0+t_episode*n}", f"tend = {10.0+t_episode*(n+1)}")
    with open("input.nml", "w") as f:
        f.write(new_content)

    dthe=np.random.uniform(0,1,size=2)
    ome=ome0 + dthe[0]*np.arange(nt_limit)*dt_limit/t_episode*np.abs(ome0)/10
    mu=mu0 + dthe[1]*np.arange(nt_limit)*dt_limit/t_episode*np.abs(mu0)/10

    x=orbit(ome,mu,nt_limit,dt_limit,x0=x0)
    ome0,mu0,x0=ome[-1],mu[-1],x[-1]

    ts=10.+t_episode*n+np.arange(0.,t_episode+dt,dt,dtype=np.float64)
    with open("omega.txt", "w") as f:
        omega=x[::100,1]
        for i in range(len(ts)):
            f.write(f'{ts[i]} {omega[i]} \n')

    result = subprocess.run(['../../src/a.out'], capture_output=True, text=True)
    print(n)
    
    df = pd.read_csv('./axke.dat', sep="\s+",header=None, skiprows=1)
    tmp=df.iloc[:,0].values
    print('mean:',df.iloc[:,1].values.mean())

    plt.plot(ts,omega)
    #plt.xlim([len(x)*0.001-2*per,len(x)*0.001])
    plt.grid()
    plt.show()

    plt.plot(df.iloc[:,0].values,df.iloc[:,1].values)
    plt.show()


    result = subprocess.run(['mv','axke.dat',f'axke_{str(n).zfill(3)}.dat'])
    result = subprocess.run(['mv','torque.dat',f'torque_{str(n).zfill(3)}.dat'])
    result = subprocess.run(['mv','omega.txt',f'omega_{str(n).zfill(3)}.zfill(3).txt'])

In [ ]:
from scipy.interpolate import interp1d
dt_limit=0.0001
nt_limit=round(t_episode/dt_limit)+1

def episode(ome0,mu0,x0,n,dthe):
    with open("input.nml", "r") as f:
        content = f.read()
    new_content = content.replace(f"tend = {10.0}", f"tend = {t_episode*(n+1)}")
    with open("input.nml", "w") as f:
        f.write(new_content)

    
    ome=ome0 + dthe[0]*np.arange(nt_limit)*dt_limit/t_episode*alpha_t0
    mu=mu0 + dthe[1]*np.arange(nt_limit)*dt_limit/t_episode*alpha_t0

    x=orbit(np.exp(ome),np.exp(mu),nt_limit,dt_limit,x0=x0)
    ome0,mu0,x0=ome[-1],mu[-1],x[-1]

    ts=t_episode*n+np.arange(0.,t_episode+dt/10,dt,dtype=np.float64)
    with open("omega.txt", "w") as f:
        omega=x[::100,1]
        for i in range(len(ts)):
            f.write(f'{ts[i]} {omega[i]} \n')

    result = subprocess.run(['../../src/a.out'], capture_output=True, text=True)

    obs=x[::100]
    act=obs[:,1:2]
    df = pd.read_csv('./axke.dat', sep="\s+",header=None, skiprows=1)
    f=interp1d(df.iloc[:,0].values,df.iloc[:,1].values,fill_value='extrapolate',kind="cubic")
    rew=f(ts).reshape(-1,1)
    the=np.stack([ome[::100],mu[::100]],axis=1)

    result = subprocess.run(['mv','axke.dat',f'axke_{str(n).zfill(3)}.dat'])
    result = subprocess.run(['mv','torque.dat',f'torque_{str(n).zfill(3)}.dat'])
    result = subprocess.run(['mv','omega.txt',f'omega_{str(n).zfill(3)}.txt'])

    return obs,act,rew,the

In [ ]:
ome0,mu0,x0=the0[0],the0[1],[0,2*np.pi]
for n in range(0):
    result = subprocess.run(['cp','input1.nml','input.nml'])
    obs,act,rew,the=episode(ome0,mu0,x0,n)
    ome0,mu0,x0=the[-1,0],the[-1,1],obs[-1]
    plt.plot(obs[:,0])
    plt.plot(obs[:,1])
    plt.show()

In [ ]:
# module
import sys
import struct
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
import sklearn.pipeline
import sklearn.preprocessing
from sklearn.kernel_approximation import RBFSampler
import time
from numba import jit, f8, i8, b1, void
from sklearn.linear_model import LinearRegression

max_batch = 10000       #number of episode
n_step = 100           # n_step TD error, dimensionless time = 1
dim_obs = 2            # dimension of observe
dim_act  = 1           # dimension of action
dim_the   = 2    # dimension of theta
gam = 0.9              # discount rate

#== important parameters ==#
#batch_size = 5000            # num. of time steps with in a episode
alpha_t0 = 1e1 #/1000        # learning rate of Actor
alpha_t = alpha_t0
num_train_episode = 10       # num. of episodes for Critic

gamma_list = [0.1,1.,10]
num_components=100
scaler_observe = sklearn.preprocessing.StandardScaler()
feature = sklearn.pipeline.FeatureUnion([
    (str(i), RBFSampler(gamma=gamma_list[i],
                       n_components=num_components))
    for i in range(len(gamma_list))
])

def minimize_TDerror5(obs_np,the_np,act_np,rew_np,phi_imp_np,n_step,gam,rcond=1e-15):
#====preparing samples at t and t+1 ===#
    obs0,the0,act0,phi0=obs_np[:-n_step],the_np[:-n_step],act_np[:-n_step],phi_imp_np[:-n_step]
    obs1,the1,act1,phi1=obs_np[n_step:],the_np[n_step:],act_np[n_step:],phi_imp_np[n_step:]


#===setting of linear EQ ===#
    p=gam*phi1-phi0
    
    rew0=np.zeros(len(rew_np)-n_step)
    gams=gam**(np.arange(n_step)/n_step)
    for n in range(len(rew0)):
        rew0[n]=np.sum(rew_np[n:n+n_step]*gams) 
    A = np.matmul(p.T,p)
    Y =-np.matmul(rew0,p)

#=== solve linear EQ ===#
    #X=np.linalg.solve(A,Y)
    A2=np.linalg.pinv(A,rcond=rcond, hermitian=True)
    X=np.matmul(A2,Y)

#=== TD error ===#
    TDerr=(((np.matmul(p,X)+rew0)**2).mean()/(np.matmul(phi0,X)**2).mean())**0.5
    #print(TDerr)
    
#=== value function ===#    
    Value=np.matmul(phi0,X)
    
    #== theta ==#
    one=np.ones(the0.shape[0]).reshape(-1,1)
    the0_expand=np.concatenate([the0, one], 1)

    #== setting of linear EQ ==#
    A = np.matmul(the0_expand.T,the0_expand)
    Y =np.matmul(Value,the0_expand)

    #=== solve linear EQ ===#
    A2=np.linalg.pinv(A,rcond=rcond, hermitian=True)
    X=np.matmul(A2,Y)
    dthe=X[:the0.shape[1]]*(1-gam**(1./n_step)) 
    
    return dthe,TDerr #w_weight,v_weight#,A,Y,X,A2

In [ ]:
#=== initialization of policy parameters ===#
np.random.seed(seed=0)
the=np.zeros(dim_the)#np.random.normal(loc=0,scale=1e-1,size=dim_the)
dthe=np.zeros(dim_the)


#===obsarvable and action===#
#obs     = np.array([lift,Xc[0][1]])     # observe_t
#act     = np.zeros(dim_act)     # action_t

# == history == #
J_total_history = []         # average reword within a episode
the_train_history = []       
obs_train_history = []
act_train_history = []
rew_train_history = []

total_step = 0               # total step of DNS 


TDerr,TDerr2,TDerr3 = 0,0,0
DNS_time ,Of_time,data_time,np_time,feature_time=0,0,0,0,0
Qfit1_time,Qfit2_time,Qfit3_time=0,0,0
gradient_time,adjust_time,update_time,plot_time,pickle_time=0,0,0,0,0

ome0,mu0,x0=the0[0],the0[1],[0,2*np.pi]
result = subprocess.run(['cp','input0.nml','input.nml'])
obs,act,rew,the=episode(ome0,mu0,x0,0,[0,0])
ome0,mu0,x0=the[-1,0],the[-1,1],obs[-1]

#=== learning starts ===#
for i_episode in range(1, max_batch + 1):  
    total1 = time.time()##########################################################
    t1=time.time()##########################################################
        
    #=== Initialize for DNS ===#
    steps = 0 
    
    #========== Perform DNS ==========#
    if i_episode<=10:
        dthe=np.random.uniform(-1,1,size=2)/1000
        #dthe[1]=dthe[1]/1000

    result = subprocess.run(['cp','input1.nml','input.nml'])
    obs,act,rew,the=episode(ome0,mu0,x0,i_episode,dthe)
    ome0,mu0,x0=the[-1,0],the[-1,1],obs[-1]
    obs_train_history.append(obs[:-1])
    act_train_history.append(act[:-1])
    rew_train_history.append(rew[:-1])
    the_train_history.append(the[:-1])
                
    t2=time.time()##########################################################
    DNS_time=t2-t1##########################################################
    
#=== Object function ===#
    t1=time.time()##########################################################
    J=np.mean(np.concatenate(rew_train_history[-num_train_episode:],axis=0))
    J_total_history.append(J)
    t2=time.time()##########################################################
    Of_time=t2-t1##########################################################
    
#==== training data ====#    
    #t1=time.time()##########################################################
    #if len(obs_train_history)==batch_size*(num_train_episode+1):
    del obs_train_history[:-num_train_episode]
    del rew_train_history[:-num_train_episode]
    del the_train_history[:-num_train_episode]
    del act_train_history[:-num_train_episode]
    #t2=time.time()##########################################################
    #data_time=t2-t1##########################################################
    

#========= start of RL =========#    
    if i_episode >= num_train_episode+1:
        t1=time.time()##########################################################
        obs_np=np.concatenate(obs_train_history,axis=0)#[-batch_size*num_train_episode:])
        rew_np=np.concatenate(rew_train_history,axis=0)#[-batch_size*num_train_episode:])
        the_np=np.concatenate(the_train_history,axis=0)#[-batch_size*num_train_episode:])
        act_np=np.concatenate(act_train_history,axis=0)#[-batch_size*num_train_episode:])
        t2=time.time()##########################################################
        np_time=t2-t1##########################################################

#========= Feature values =========# 
        t1=time.time()##########################################################
        #print('episode for phi',obs_np[-batch_size:].shape)

        obs_np=obs_np/10
        obs_the_np=np.concatenate([obs_np, the_np], 1)
        #obs_the_scale_np = (obs_the_np - obs_the_np.mean(0)) / obs_the_np.std(0)
        phi_imp_np = feature.fit_transform(obs_the_np)
        t2=time.time()##########################################################
        feature_time=t2-t1##########################################################
    
#==== fitting of value function====#         
        t1=time.time()##########################################################
        dthe,TDerr=minimize_TDerror5(obs_np,the_np,act_np,rew_np,phi_imp_np,n_step,gam)
        #dthe[0]=dthe[0]/10
        #dthe[1]=dthe[1]/1000
        t2=time.time()##########################################################
        Qfit3_time=t2-t1##########################################################
        
#=== plot of obserbation and objective function===#
    t1=time.time()##########################################################
    plt.plot(np.concatenate(obs_train_history,axis=0))
    plt.xlabel('step')
    plt.ylabel('obsevables')
    plt.grid()
    plt.show()
    plt.plot(J_total_history)
    plt.xlabel('episode')
    plt.ylabel('speed')
    plt.grid()
    plt.show()
    t2=time.time()##########################################################
    plot_time=t2-t1##########################################################

#=== save of trajectries ===#
    t1=time.time()##########################################################
    with open('obs.pickle', 'wb') as f:
        pickle.dump(obs_train_history,f)
    with open('act.pickle', 'wb') as f:
        pickle.dump(act_train_history,f)
    with open('J.pickle', 'wb') as f:
        pickle.dump(J_total_history,f)
    with open('the.pickle', 'wb') as f:
        pickle.dump(the_train_history,f)
    with open('rew.pickle', 'wb') as f:
        pickle.dump(rew_train_history,f)
    t2=time.time()##########################################################
    pickle_time=t2-t1##########################################################
    
#=== display ===#        
    print("#====================",i_episode, " episode ====================#")
    print("J       = ", J           )
    print("dtheta  = ", dthe        )
    print("theta   = ", the         )
    print("TDerr   = ", TDerr       )
    print("alpha_t = ", alpha_t     )
    #print("w       = ", w_weight)
    #print("v       = ", v_weight)

    total2 = time.time()##########################################################
    print("#=============================================#")
    print("TOTAL TIME per minibatch : ", total2-total1  )